In [ ]:
pip install ortools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 27.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.5 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatib

In [ ]:
# =========================================================
# CELL 1: IMPORTS
# =========================================================
from ortools.sat.python import cp_model
import pandas as pd

In [ ]:
# =========================================================
# CELL 2: LOAD CSV FILES
# =========================================================
subjects_df = pd.read_csv("subjects.csv")
faculty_df = pd.read_csv("faculty.csv")

print("\n--- SUBJECTS DATA ---")
print(subjects_df)

print("\n--- FACULTY DATA ---")
print(faculty_df)


--- SUBJECTS DATA ---
  subject_code               name  credits  Total_hours  failure rate
0        CS101    Data Structures        4            5            15
1        CS102               DBMS        3            4            30
2        CS103  Operating Systems        4            5            20
3        CS104         Networking        3            4            23
4        CS105                 AI        3            4            35
5        CS106                PPS        2            2            16
6        CS107  computer Graphics        3            4             5
7        CS108        Basics of C        3            5            43

--- FACULTY DATA ---
  faculty_id        name subjects
0        F01     Dr. Rao    CS101
1        F02  Ms. Anjali    CS102
2        F03   Mr. Kiran    CS103
3        F04   Dr. Reddy    CS104
4        F05   Ms. Priya    CS105
5        F06    Dr. John    CS106
6        F07    Ms.Priya    CS107
7        F08    Ms.Divya    CS108


In [ ]:
# =========================================================
# CELL 3: BASIC VALIDATIONS
# =========================================================

# Rule 1: Each subject total hours must not exceed 5
invalid_subjects = subjects_df[subjects_df["Total_hours"] > 5]
if not invalid_subjects.empty:
    raise ValueError(
        f"❌ ERROR: Subjects with Total_hours > 5 detected:\n{invalid_subjects}"
    )

In [ ]:
# Rule 2: Total weekly hours must fit in 7 slots × 5 days
TOTAL_AVAILABLE_SLOTS = 7 * 5
TOTAL_REQUIRED_HOURS = subjects_df["Total_hours"].sum()

print(f"\nTotal Required Hours: {TOTAL_REQUIRED_HOURS}")
print(f"Total Available Slots: {TOTAL_AVAILABLE_SLOTS}")

if TOTAL_REQUIRED_HOURS > TOTAL_AVAILABLE_SLOTS:
    raise ValueError(
        "❌ ERROR: Total subject hours exceed available weekly slots."
    )

print("\nHour validations passed.")


Total Required Hours: 33
Total Available Slots: 35

Hour validations passed.


In [ ]:
# =========================================================
# CELL 4: CALCULATE & NORMALIZE SUBJECT DIFFICULTY (0–1)
# =========================================================
subjects_df["raw_difficulty"] = (
    subjects_df["credits"] * subjects_df["failure rate"]
)

max_diff = subjects_df["raw_difficulty"].max()
min_diff = subjects_df["raw_difficulty"].min()

subjects_df["difficulty"] = (
    (subjects_df["raw_difficulty"] - min_diff) /
    (max_diff - min_diff)
)

print("\n---  SUBJECT DIFFICULTY  ---")
print(
    subjects_df[
        ["subject_code", "credits", "failure rate", "difficulty"]
    ]
)



---  SUBJECT DIFFICULTY  ---
  subject_code  credits  failure rate  difficulty
0        CS101        4            15    0.394737
1        CS102        3            30    0.657895
2        CS103        4            20    0.570175
3        CS104        3            23    0.473684
4        CS105        3            35    0.789474
5        CS106        2            16    0.149123
6        CS107        3             5    0.000000
7        CS108        3            43    1.000000


In [ ]:
# =========================================================
# CELL 5: SELECT TOP 2 DIFFICULT SUBJECTS
# =========================================================
subjects_df = subjects_df.sort_values(
    by="difficulty", ascending=False
)

difficult_subjects = subjects_df.head(2)["subject_code"].tolist()

print("\nTOP 2 DIFFICULT SUBJECTS (MORNING ONLY):")
print(difficult_subjects)


TOP 2 DIFFICULT SUBJECTS (MORNING ONLY):
['CS108', 'CS105']


In [ ]:
# =========================================================
# CELL 6: MODEL PARAMETERS
# =========================================================
DAYS = range(5)          # Mon–Fri
SLOTS = range(8)         # 0–7
LUNCH_SLOT = 4
SECTIONS = range(4)

subjects = list(subjects_df["subject_code"])
subject_hours = dict(
    zip(subjects_df["subject_code"], subjects_df["Total_hours"])
)


In [ ]:
# =========================================================
# CELL 7: FACULTY MAP
# =========================================================
faculty_of = {}
for _, row in faculty_df.iterrows():
    for s in row["subjects"].split(","):
        faculty_of[s.strip()] = row["faculty_id"]

faculties = set(faculty_of.values())

In [ ]:
# =========================================================
# CELL 8: CREATE MODEL
# =========================================================
model = cp_model.CpModel()

X = {}
for sec in SECTIONS:
    for d in DAYS:
        for s in SLOTS:
            for sub in subjects:
                X[(sec, d, s, sub)] = model.NewBoolVar(
                    f"x_s{sec}_d{d}_t{s}_{sub}"
                )

In [ ]:
# =========================================================
# CELL 9: CONSTRAINTS (UNCHANGED)
# =========================================================

# One subject per slot (lunch free)
for sec in SECTIONS:
    for d in DAYS:
        for s in SLOTS:
            if s == LUNCH_SLOT:
                model.Add(
                    sum(X[(sec, d, s, sub)] for sub in subjects) == 0
                )
            else:
                model.Add(
                    sum(X[(sec, d, s, sub)] for sub in subjects) <= 1
                )

# Subject at most once per day
for sec in SECTIONS:
    for d in DAYS:
        for sub in subjects:
            model.Add(
                sum(X[(sec, d, s, sub)] for s in SLOTS) <= 1
            )

# Weekly hours exact
for sec in SECTIONS:
    for sub in subjects:
        model.Add(
            sum(
                X[(sec, d, s, sub)]
                for d in DAYS for s in SLOTS
            ) == subject_hours[sub]
        )

# Faculty clash
for d in DAYS:
    for s in SLOTS:
        if s == LUNCH_SLOT:
            continue
        for fac in faculties:
            model.Add(
                sum(
                    X[(sec, d, s, sub)]
                    for sec in SECTIONS
                    for sub in subjects
                    if faculty_of[sub] == fac
                ) <= 1
            )

In [ ]:
# =========================================================
# CELL 10: MORNING-ONLY CONSTRAINT FOR DIFFICULT SUBJECTS
# =========================================================
for sub in difficult_subjects:
    for sec in SECTIONS:
        for d in DAYS:
            for s in range(LUNCH_SLOT + 1, 8):
                model.Add(X[(sec, d, s, sub)] == 0)

In [ ]:
# =========================================================
# CELL 11: SOLVE
# =========================================================
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30
solver.parameters.num_search_workers = 8

status = solver.Solve(model)

In [ ]:
# =========================================================
# CELL 12: PERFECT TABULAR OUTPUT WITH | SEPARATORS
# =========================================================
DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri"]
COL_WIDTH = 6  # Perfect for CS101, LUNCH, dashes

def print_timetable_section(sec, solver, X, subjects, SLOTS, LUNCH_SLOT, DAYS):
    """Print perfect aligned table using only | separators"""

    # Header banner
    banner_width = 7 + len(SLOTS) * (COL_WIDTH + 3)
    print(f"\n{'=' * banner_width}")
    print(f"{' SECTION ' + str(sec + 1):^{banner_width}}")
    print(f"{'=' * banner_width}")

    # Column header - FIXED f-string syntax
    header = "Day  |"
    for i in SLOTS:
        header += f"S{i:^6} |"
    print(header.rstrip("|"))
    print("-" * len(header.rstrip("|")))

    # Data rows
    for d in DAYS:
        row = f"{DAY_NAMES[d]:^4} |"
        for s in SLOTS:
            if s == LUNCH_SLOT:
                cell = "LUNCH"
            else:
                cell = "FREE"
                for sub in subjects:
                    if solver.Value(X[(sec, d, s, sub)]) == 1:
                        cell = sub
                        break
            row += f" {cell:^{COL_WIDTH}} |"
        print(row.rstrip("|"))
        print("-" * len(row.rstrip("|")))

    print(f"{'=' * banner_width}")

if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("\n No valid timetable exists.")
else:
    print(" TIMETABLE GENERATED!")

    # Print all sections
    for sec in SECTIONS:
        print_timetable_section(sec, solver, X, subjects, SLOTS, LUNCH_SLOT, DAYS)

    # Summary
    #print("\n" + "="*40)
    #print("="*40)


 TIMETABLE GENERATED!

                                   SECTION 1                                   
Day  |S  0    |S  1    |S  2    |S  3    |S  4    |S  5    |S  6    |S  7    
-----------------------------------------------------------------------------
Mon  | CS101  | CS106  | CS108  | CS105  | LUNCH  | CS107  | CS104  | CS103  
-----------------------------------------------------------------------------
Tue  | CS105  | CS103  | CS108  |  FREE  | LUNCH  | CS101  | CS107  | CS102  
-----------------------------------------------------------------------------
Wed  | CS105  | CS101  | CS108  | CS103  | LUNCH  | CS107  | CS104  | CS102  
-----------------------------------------------------------------------------
Thu  | CS105  | CS102  | CS108  | CS107  | LUNCH  | CS101  | CS103  | CS104  
-----------------------------------------------------------------------------
Fri  | CS103  | CS101  | CS106  | CS108  | LUNCH  | CS104  |  FREE  | CS102  
---------------------------------------

In [ ]:
!pip install reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.9 MB/s eta 0:00:00


In [ ]:
from reportlab.platypus import (
    SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
)
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet


In [ ]:
def generate_timetable_pdf(solver, X, subjects, filename="Timetable_All_Sections.pdf"):
    doc = SimpleDocTemplate(
        filename,
        pagesize=A4,
        rightMargin=30,
        leftMargin=30,
        topMargin=30,
        bottomMargin=30
    )

    styles = getSampleStyleSheet()
    elements = []

    DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri"]
    SLOT_HEADERS = ["Day", "1", "2", "3", "4", "LUNCH", "6", "7", "8"]

    for sec in SECTIONS:
        # Section Title
        elements.append(
            Paragraph(f"<b>SECTION {sec + 1} TIMETABLE</b>", styles["Title"])
        )
        elements.append(Spacer(1, 12))

        table_data = [SLOT_HEADERS]

        for d in DAYS:
            row = [DAY_NAMES[d]]
            for s in SLOTS:
                if s == LUNCH_SLOT:
                    row.append("LUNCH")
                else:
                    val = "--"
                    for sub in subjects:
                        if solver.Value(X[(sec, d, s, sub)]) == 1:
                            val = sub
                    row.append(val)
            table_data.append(row)

        table = Table(table_data, repeatRows=1)

        table.setStyle(TableStyle([
            ("GRID", (0, 0), (-1, -1), 1, colors.black),
            ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
            ("ALIGN", (1, 1), (-1, -1), "CENTER"),
            ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
            ("TOPPADDING", (0, 0), (-1, -1), 6),
        ]))

        elements.append(table)
        elements.append(Spacer(1, 24))

    doc.build(elements)
    print(f"\nPDF generated successfully: {filename}")


In [ ]:
if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    generate_timetable_pdf(
        solver,
        X,
        subjects,
        filename="Sections_timetable.pdf"
    )



PDF generated successfully: Sections_timetable.pdf


**FACULTY TIMETABLES**

In [ ]:
# =========================================================
# CELL 13: FACULTY NAME MAP
# =========================================================
faculty_name_map = dict(
    zip(faculty_df["faculty_id"], faculty_df["name"])
)


In [ ]:
# =========================================================
# CELL 14: BUILD FACULTY TIMETABLE
# =========================================================
faculty_tt = {
    fac: {
        d: {s: "" for s in SLOTS}
        for d in DAYS
    }
    for fac in faculties
}

for sec in SECTIONS:
    for d in DAYS:
        for s in SLOTS:
            if s == LUNCH_SLOT:
                continue
            for sub in subjects:
                if solver.Value(X[(sec, d, s, sub)]) == 1:
                    fac = faculty_of[sub]
                    faculty_tt[fac][d][s] = f"SEC-{sec+1}"


In [ ]:
# =========================================================
# CELL 15: PRINT FACULTY TIMETABLES
# =========================================================
COL_WIDTH = 12
DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri"]

for fac in sorted(faculty_tt.keys()):
    fac_name = faculty_name_map.get(fac, "")
    print(f"\nFACULTY : {fac}  ({fac_name})")
    print("-" * (COL_WIDTH * 9))

    # Header
    header = f"{'Day':<{COL_WIDTH}}"
    for s in SLOTS:
        slot_label = "LUNCH" if s == LUNCH_SLOT else f"S{s+1}"
        header += f"{slot_label:^{COL_WIDTH}}"
    print(header)
    print("-" * (COL_WIDTH * 9))

    # Rows
    for d in DAYS:
        row = f"{DAY_NAMES[d]:<{COL_WIDTH}}"
        for s in SLOTS:
            if s == LUNCH_SLOT:
                cell = "LUNCH"
            else:
                cell = faculty_tt[fac][d][s] or "--"
            row += f"{cell:^{COL_WIDTH}}"
        print(row)

    print("-" * (COL_WIDTH * 9))



FACULTY : F01  (Dr. Rao)
------------------------------------------------------------------------------------------------------------
Day              S1          S2          S3          S4        LUNCH         S6          S7          S8     
------------------------------------------------------------------------------------------------------------
Mon            SEC-1         --        SEC-4       SEC-2       LUNCH         --        SEC-3         --     
Tue              --          --        SEC-4         --        LUNCH       SEC-1       SEC-3       SEC-2    
Wed            SEC-2       SEC-1         --          --        LUNCH         --        SEC-3       SEC-4    
Thu            SEC-3         --          --        SEC-2       LUNCH       SEC-1         --        SEC-4    
Fri              --        SEC-1         --        SEC-2       LUNCH       SEC-4       SEC-3         --     
------------------------------------------------------------------------------------------------------

In [ ]:
# =========================================================
# FACULTY TIMETABLES → SINGLE PDF
# =========================================================

import pandas as pd
from reportlab.platypus import (
    SimpleDocTemplate, Table, TableStyle,
    Paragraph, Spacer, PageBreak
)
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

# ---------------------------------------------------------
# LOAD FACULTY FILE
# ---------------------------------------------------------
faculty_df = pd.read_csv("faculty.csv")   # ensure file exists

# ---------------------------------------------------------
# FACULTY → SUBJECT MAP
# ---------------------------------------------------------
faculty_of = {}
for _, row in faculty_df.iterrows():
    for sub in row["subjects"].split(","):
        faculty_of[sub.strip()] = row["faculty_id"]

# Optional faculty names
faculty_name_map = {}
if "name" in faculty_df.columns:
    faculty_name_map = dict(
        zip(faculty_df["faculty_id"], faculty_df["name"])
    )

# ---------------------------------------------------------
# BUILD FACULTY TIMETABLE STRUCTURE
# ---------------------------------------------------------
faculty_tt = {
    fac: [[ "" for _ in SLOTS ] for _ in DAYS]
    for fac in faculty_of.values()
}

for sec in SECTIONS:
    for d in DAYS:
        for s in SLOTS:
            if s == LUNCH_SLOT:
                continue
            for sub in subjects:
                if solver.Value(X[(sec, d, s, sub)]) == 1:
                    fac = faculty_of[sub]
                    faculty_tt[fac][d][s] = f"SEC {sec + 1}"

# ---------------------------------------------------------
# CREATE PDF
# ---------------------------------------------------------
pdf_path = "faculty_timetables.pdf"

doc = SimpleDocTemplate(pdf_path, pagesize=A4)
styles = getSampleStyleSheet()
elements = []

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri"]
COL_WIDTH = 60

for fac in sorted(faculty_tt.keys()):
    fac_name = faculty_name_map.get(fac, "")

    elements.append(
        Paragraph(
            f"<b>Faculty ID:</b> {fac} &nbsp;&nbsp; "
            f"<b>Name:</b> {fac_name}",
            styles["Title"]
        )
    )
    elements.append(Spacer(1, 12))

    table_data = [["Day"] + [
        "LUNCH" if s == LUNCH_SLOT else f"S{s + 1}"
        for s in SLOTS
    ]]

    for d in DAYS:
        row = [DAY_NAMES[d]]
        for s in SLOTS:
            if s == LUNCH_SLOT:
                row.append("LUNCH")
            else:
                row.append(faculty_tt[fac][d][s] or "--")
        table_data.append(row)

    table = Table(table_data, colWidths=[COL_WIDTH] * 9)
    table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 1, colors.black),
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("FONT", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("ALIGN", (1, 1), (-1, -1), "CENTER"),
        ("ALIGN", (0, 0), (0, -1), "CENTER"),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))

    elements.append(table)
    elements.append(PageBreak())

doc.build(elements)

print("✅ Faculty timetables PDF generated successfully!")
print("📄 File:", pdf_path)


✅ Faculty timetables PDF generated successfully!
📄 File: faculty_timetables.pdf


 **Constraints Used in the Timetable Scheduling Project**

**1. Time Structure Constraints**

* The timetable spans **5 working days** (Monday to Friday).
* Each day consists of **8 time slots**.
* **Slot 4 is fixed as the LUNCH break** and no subject can be scheduled   in this slot.
* Only **7 teaching slots per day** are available for scheduling.

---

**2. Section Constraints**

* The system generates timetables for **multiple sections**.
* Each section has an **independent timetable**, but shares faculty resources.

---

**3. Slot Allocation Constraints**

* **At most one subject** can be scheduled in a single time slot for a section.
* A subject **cannot appear more than once per day per section**.
* Empty slots (--) are allowed when all constraints cannot fill the timetable fully.

---
**4. Subject Hour Constraints**

* Each subject must be scheduled for **exactly its required weekly hours** as specified in subjects.csv.
---
**5. Validation constraint:**

* If any subject has Total_hours > 5, the model raises an error (since only one occurrence per day is allowed).
---
**6. Validation constraint:**

* The **sum of all subject weekly hours** must not exceed 7 slots * 5 days = 35.
---
**7. Faculty Assignment Constraints**

* Each subject is taught by **exactly one faculty member**.
* A faculty member **cannot be assigned to more than one section at the same time slot**.
* Faculty clashes are prevented **across all sections, days, and slots**.

---

**8. Difficulty-Based Academic Constraints**

* Subject difficulty is calculated using:
    Difficulty = Credits*Failure Rate
* Difficulty values are **normalized to the range [0, 1]**.
* The **top 2 most difficult subjects** are automatically identified.
* These difficult subjects are **restricted to morning (pre-lunch) slots only**.

---


**9. Model Correctness Constraints**
* A timetable is considered valid **only if all constraints are satisfied simultaneously**.
* If no feasible solution exists, the solver explicitly reports:

    ❌ No valid timetable exists
---
**10. Output & Representation Constraints**

* Section timetables are printed in **clear tabular format**.
* Faculty timetables show:
  a. Faculty ID and Name
  b. **Only section numbers (no subject codes)**.
* All faculty timetables are exported into **a single consolidated PDF**.
* Subjects and faculty mappings are validated before model creation.
* Any missing or inconsistent data results in **immediate termination with error**.


**ALGORITHMS**

The project uses a Constraint Programming (CP) approach, specifically:
* CP-SAT Solver (Constraint Programming - Satisfiability) provided by Google OR-Tools

Why CP-SAT?

Timetable generation is a combinatorial optimization problem, not a prediction problem.

we are not learning from past data.
we are searching for a feasible assignment that satisfies strict rules.

 Constraint satisfaction + logical inference

**What Actually Happens**
* define variables
* define constraints

**The CP-SAT solver:**

* Explores combinations
* Prunes impossible assignments
* Uses backtracking + SAT reasoning
* Stops when a feasible or optimal solution is found

**Decision Variables (Core of the Model)**
* Decision variable is:
* Xsec,d,s,sub =
  
  1 if subject(sub) is scheduled in section(sec) on day(d) at slot(s)
  
  0 otherwise X sec,d,s,sub




**Mathematical Formulation of Constraints**
*  One Subject per Slot (Except Lunch)
*  Subject Appears at Most Once per Day
*  Weekly Hours Constraint (Exact)
*  Faculty clash constraint(A faculty member can teach only one section at a time)

 **Difficulty-Based Morning Constraint**

* You compute difficulty as:
Raw Difficulty = Credits*FailureRate

* Then normalize:
Difficulty= Raw Difficulty-min/max-min

This forces:
* Difficult subjects only in High attention / morning slots.

**Data Validation Violation**
* Total_hours_per_subject <= 5
* Total_hours_per_week <= 35 if violated no timetable exists.

**How the Solver Finds a Solution**
Internally, CP-SAT uses:
* SAT (Satisfiability) solving
* Constraint propagation
* Domain pruning
* Intelligent backtracking
* Parallel search (num_search_workers = 8)

**The solver:**
* Assigns values to variables
* Checks constraint violations
* Backtracks if needed
Stops when all constraints are satisfied.

It generates timetable only if feasible and optimal solution was found and all constraints are satisfied
* No faculty clashes
* Exact weekly hours
* Lunch preserved
* Difficult subjects placed correctly.

**Section TimeTable Generation**
* Scan each slot
* Detect the assigned subject
* Build a tabular timetable using Pandas
* This is post-processing, not optimization.

**Faculty Timetable Generation**
* Read section timetables
* Map subject → faculty